# Data Cleaning Pipeline: AI Occupational Exposure Scores

This notebook prepares data for a machine learning project predicting AI Occupational Exposure (AIOE) scores. We merge occupation-level features from O\*NET with AIOE target variables and BLS demographic/wage data, then clean and standardize everything for modeling.

**Target variables (y):** AIOE, Language Modeling AIOE (LMOE), Image Generation AIOE (IGOE)  
**Predictor variables (X):** O\*NET skills, abilities, work activities, work context, education/training/experience, job zones  
**Additional predictors:** BLS demographic shares (pct_women, pct_black, pct_hispanic, pct_asian), median wages, total employment

**Datasets:**
- `occupation-level AIOE scores.xlsx` — AIOE scores by occupation
- `Language Modeling AIOE and AIIE.xlsx` — LMOE scores by occupation
- `Image Generation AIOE and AIIE.xlsx` — IGOE scores by occupation
- O\*NET files: Skills, Abilities, Work Activities, Work Context, Education/Training/Experience, Job Zones
- `BLS Occupational Employment and Wage Statistics.xlsx` — wages and employment
- `BLS Current Population Survey.xlsx` — demographic shares

In [67]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Base paths
DATA_DIR = "data"
AIOE_DIR = f"{DATA_DIR}/AIOE Data"
ONET_DIR = f"{DATA_DIR}/O*NET Data"

# Cleaning log tracker
cleaning_log = []

def log(msg):
    """Print and record a cleaning step."""
    print(msg)
    cleaning_log.append(msg)

def inspect(df, name="Dataset"):
    """Print shape, head, missing values, and basic stats."""
    print(f"\n{'='*60}")
    print(f"{name}: shape = {df.shape}")
    print(f"\n--- Head ---")
    display(df.head())
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\n--- Missing Values ---")
        print(missing[missing > 0])
    else:
        print(f"\nNo missing values.")
    print(f"\n--- Descriptive Statistics ---")
    display(df.describe(include='all').T)

def standardize_soc(code):
    """Truncate O*NET 8-digit SOC code (e.g. 11-1011.00) to 6-digit (11-1011)."""
    s = str(code).strip()
    if '.' in s:
        s = s.split('.')[0]
    return s

## Step 1: Load and Inspect Each Dataset

We load every dataset and print its shape, first rows, missing value counts, and descriptive statistics. This gives us a baseline understanding of what we're working with before any transformations.

### 1a. Target Variables — AIOE, LMOE, IGOE Scores

In [68]:
# Load AIOE scores
aioe = pd.read_excel(f"occupation-level AIOE scores.xlsx", sheet_name="Appendix A")
inspect(aioe, "AIOE Scores")


AIOE Scores: shape = (774, 3)

--- Head ---


,SOC Code,Occupation Title,AIOE
0,11-1011,Chief Executives,1.334246
1,11-1021,General and Operations Managers,0.574877
2,11-2011,Advertising and Promotions Managers,1.294387
3,11-2021,Marketing Managers,1.315032
4,11-2022,Sales Managers,1.266280



No missing values.

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SOC Code,774,774,53-7121,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Occupation Title,774,774,"Tank Car, Truck, and Ship Loaders",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AIOE,774.0,NaN,NaN,NaN,-0.0,1.0,-2.669758,-0.864545,-0.051193,1.014434,1.527667


In [69]:
# Load Language Modeling AIOE (LMOE)
lmoe = pd.read_excel(f"Language Modeling AIOE and AIIE.xlsx", sheet_name="LM AIOE")
inspect(lmoe, "Language Modeling AIOE")


Language Modeling AIOE: shape = (774, 3)

--- Head ---


,SOC Code,Occupation Title,Language Modeling AIOE
0,11-1011,Chief Executives,1.308912
1,11-1021,General and Operations Managers,0.677615
2,11-2011,Advertising and Promotions Managers,1.217224
3,11-2021,Marketing Managers,1.203774
4,11-2022,Sales Managers,1.293821



No missing values.

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SOC Code,774,774,53-7121,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Occupation Title,774,774,"Tank Car, Truck, and Ship Loaders",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Language Modeling AIOE,774.0,NaN,NaN,NaN,0.0,1.0,-1.853906,-0.915624,-0.094706,0.951072,1.925633


In [70]:
# Load Image Generation AIOE (IGOE)
igoe = pd.read_excel(f"Image Generation AIOE and AIIE.xlsx", sheet_name="IG AIOE")
inspect(igoe, "Image Generation AIOE")


Image Generation AIOE: shape = (774, 3)

--- Head ---


,SOC Code,Occupation Title,Image Generation AIOE
0,11-1011,Chief Executives,1.309602
1,11-1021,General and Operations Managers,0.231884
2,11-2011,Advertising and Promotions Managers,1.614598
3,11-2021,Marketing Managers,1.610358
4,11-2022,Sales Managers,1.009998



No missing values.

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SOC Code,774,774,53-7121,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Occupation Title,774,774,"Tank Car, Truck, and Ship Loaders",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Image Generation AIOE,774.0,NaN,NaN,NaN,0.0,1.0,-3.654782,-0.722951,-0.021375,0.769285,2.420953


### 1b. O\*NET Feature Datasets (Predictor Variables)

These files are in **long format** — each row is one occupation-element pair with a score. We'll need to pivot them to wide format later. For now, we just load and inspect.

In [71]:
# Load all O*NET feature files
skills = pd.read_excel(f"Skills.xlsx")
abilities = pd.read_excel(f"Abilities.xlsx")
work_activities = pd.read_excel(f"Work Activities.xlsx")
work_context = pd.read_excel(f"Work Context.xlsx")
edu_train_exp = pd.read_excel(f"Education, Training, and Experience.xlsx")
job_zones = pd.read_excel(f"Job Zones.xlsx")

for name, df in [("Skills", skills), ("Abilities", abilities),
                  ("Work Activities", work_activities), ("Work Context", work_context),
                  ("Education/Training/Experience", edu_train_exp), ("Job Zones", job_zones)]:
    inspect(df, name)


Skills: shape = (62580, 15)

--- Head ---


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.1830,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst



--- Missing Values ---
Not Relevant    31290
dtype: int64

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
O*NET-SOC Code,62580,894,53-7121.00,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,62580,894,"Tank Car, Truck, and Ship Loaders",70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Element ID,62580,35,2.A.1.a,1788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Element Name,62580,35,Reading Comprehension,1788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Scale ID,62580,2,IM,31290,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Scale Name,62580,2,Importance,31290,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Value,62580.0,NaN,NaN,NaN,2.484562,1.094137,0.0,1.88,2.75,3.12,6.0
N,62580.0,NaN,NaN,NaN,8.0,0.0,8.0,8.0,8.0,8.0,8.0
Standard Error,62580.0,NaN,NaN,NaN,0.144963,0.096738,0.0,0.125,0.1637,0.189,0.7559
Lower CI Bound,62580.0,NaN,NaN,NaN,2.204853,1.103131,0.0,1.4292,2.4292,3.0,6.0



Abilities: shape = (92976, 15)

--- Head ---


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
O*NET-SOC Code,37125,878,53-7081.00,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,37125,878,Refuse and Recyclable Material Collectors,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Element ID,37125,6,2.D.1,10536,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Element Name,37125,6,Required Level of Education,10536,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Scale ID,37125,5,RL,10536,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Scale Name,37125,5,Required Level Of Education (Categories 1-12),10536,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,35998.0,NaN,NaN,NaN,5.707317,3.086156,1.0,3.0,6.0,8.0,12.0
Data Value,37125.0,NaN,NaN,NaN,9.539695,13.901753,0.0,0.0,3.57,14.29,100.0
N,37125.0,NaN,NaN,NaN,25.171152,9.474034,9.0,20.0,23.0,28.0,98.0
Standard Error,28115.0,NaN,NaN,NaN,4.704188,5.432629,0.0,0.0,2.3113,8.66905,29.7345



Job Zones: shape = (923, 5)

--- Head ---


,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst
2,11-1021.00,General and Operations Managers,4,08/2023,Analyst
3,11-1031.00,Legislators,4,06/2008,Analyst
4,11-2011.00,Advertising and Promotions Managers,4,08/2018,Analyst



No missing values.

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
O*NET-SOC Code,923,923,53-7121.00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,923,923,"Tank Car, Truck, and Ship Loaders",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Job Zone,923.0,NaN,NaN,NaN,3.218852,1.105985,2.0,2.0,3.0,4.0,5.0
Date,923,19,02/2026,331,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Domain Source,923,2,Analyst,902,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1c. BLS Datasets — Wages, Employment, and Demographics

In [72]:
# Load BLS Occupational Employment and Wage Statistics
bls_wages = pd.read_excel(f"BLS Occupational Employment and Wage Statistics.xlsx")
inspect(bls_wages, "BLS Wages & Employment")


BLS Wages & Employment: shape = (414437, 32)

--- Head ---


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN
3,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN
4,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,99.24,#,#,73710,126080,206420,#,#,NaN,NaN



--- Missing Values ---
JOBS_1000       177824
LOC_QUOTIENT    177824
PCT_TOTAL       243679
PCT_RPT         243679
ANNUAL          397752
HOURLY          413647
dtype: int64

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
AREA,414437.0,NaN,NaN,NaN,375829.46576,1133327.365209,1.0,99.0,99.0,35840.0,7800001.0
AREA_TITLE,414437,583,U.S.,177824,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AREA_TYPE,414437.0,NaN,NaN,NaN,2.769355,1.781611,1.0,1.0,2.0,4.0,6.0
PRIM_STATE,414437,55,US,177824,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NAICS,414437,414,000000,238016,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NAICS_TITLE,414437,394,Cross-industry,238016,NaN,NaN,NaN,NaN,NaN,NaN,NaN
I_GROUP,414437,9,cross-industry,238016,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OWN_CODE,414437.0,NaN,NaN,NaN,717.733228,601.771939,1.0,5.0,1235.0,1235.0,1235.0
OCC_CODE,414437,1396,13-1020,1408,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OCC_TITLE,414437,1138,General and Operations Managers,1484,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
# Load BLS Current Population Survey (demographics)
bls_cps = pd.read_excel(f"BLS Current Population Survey.xlsx")
inspect(bls_cps, "BLS CPS Demographics")


BLS CPS Demographics: shape = (374, 7)

--- Head ---


,occupation,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,"Total, 16 years and over",163493,47.1,75.7,12.7,7.5,20.0
1,Chief executives,1802,33.0,85.3,5.6,7.0,8.4
2,General and operations managers,1379,34.4,83.0,8.4,4.0,14.1
3,Legislators,18,NaN,NaN,NaN,NaN,NaN
4,Advertising and promotions managers,61,53.7,72.0,16.0,11.9,6.2



--- Missing Values ---
pct_women       14
pct_white       14
pct_black       14
pct_asian       14
pct_hispanic    14
dtype: int64

--- Descriptive Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
occupation,374,374,Other material moving workers,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_employed_thousands,374.0,NaN,NaN,NaN,861.219251,8455.184488,8.0,92.0,178.5,485.75,163493.0
pct_women,360.0,NaN,NaN,NaN,46.725833,27.110038,0.0,22.875,48.15,69.8,100.0
pct_white,360.0,NaN,NaN,NaN,76.025278,9.45435,25.1,71.4,76.85,82.1,96.6
pct_black,360.0,NaN,NaN,NaN,12.341944,7.275543,0.0,7.275,11.0,16.225,43.5
pct_asian,360.0,NaN,NaN,NaN,7.451389,7.119231,0.0,3.475,5.6,9.175,67.5
pct_hispanic,360.0,NaN,NaN,NaN,18.4725,10.633266,1.3,11.7,15.95,21.675,68.5


## Step 2: Pivot O\*NET Files from Long to Wide Format

Each O\*NET file is in long format where rows represent occupation-element-scale combinations. We pivot so that each occupation is one row, and each element (optionally combined with scale) becomes its own column.

- **Skills, Abilities, Work Activities**: Have `Scale ID` (IM = Importance, LV = Level). We combine `Element Name` + `Scale ID` to create unique column names (e.g., "Reading Comprehension_IM").
- **Work Context**: Uses `CX`/`CXP` scales with a `Category` column. We combine `Element Name` + `Scale ID` + `Category` for unique columns.
- **Education/Training/Experience**: Uses `RL` scale with `Category`. Same approach as Work Context.
- **Job Zones**: Already in wide format (one row per occupation), just needs SOC standardization.

In [74]:
def pivot_onet_simple(df, prefix, name):
    """Pivot O*NET files with Scale ID (IM/LV) — Skills, Abilities, Work Activities."""
    df = df.copy()
    # Create unique column label: ElementName_ScaleID
    df['col_name'] = prefix + "_" + df['Element Name'].astype(str) + "_" + df['Scale ID'].astype(str)
    pivoted = df.pivot_table(
        index='O*NET-SOC Code',
        columns='col_name',
        values='Data Value',
        aggfunc='mean'
    ).reset_index()
    log(f"  {name}: pivoted from {df.shape} to {pivoted.shape}")
    return pivoted

def pivot_onet_with_category(df, prefix, name):
    """Pivot O*NET files that have a Category column — Work Context, Education/Training."""
    df = df.copy()
    # Create unique column label: ElementName_ScaleID_Category
    df['col_name'] = (prefix + "_" + df['Element Name'].astype(str) + "_" +
                      df['Scale ID'].astype(str) + "_" + df['Category'].astype(str))
    pivoted = df.pivot_table(
        index='O*NET-SOC Code',
        columns='col_name',
        values='Data Value',
        aggfunc='mean'
    ).reset_index()
    log(f"  {name}: pivoted from {df.shape} to {pivoted.shape}")
    return pivoted

log("--- Step 2: Pivoting O*NET files from long to wide ---")

skills_wide = pivot_onet_simple(skills, "Skill", "Skills")
abilities_wide = pivot_onet_simple(abilities, "Ability", "Abilities")
work_act_wide = pivot_onet_simple(work_activities, "WorkAct", "Work Activities")
work_ctx_wide = pivot_onet_with_category(work_context, "WorkCtx", "Work Context")
edu_wide = pivot_onet_with_category(edu_train_exp, "EduTrExp", "Education/Training/Experience")

# Job Zones is already wide — just keep SOC code and Job Zone
job_zones_wide = job_zones[['O*NET-SOC Code', 'Job Zone']].copy()
log(f"  Job Zones: already wide, shape = {job_zones_wide.shape}")

print("\nPivot complete.")

--- Step 2: Pivoting O*NET files from long to wide ---
  Skills: pivoted from (62580, 16) to (894, 71)
  Abilities: pivoted from (92976, 16) to (894, 105)
  Work Activities: pivoted from (73308, 16) to (894, 83)
  Work Context: pivoted from (297676, 17) to (894, 339)
  Education/Training/Experience: pivoted from (37125, 16) to (878, 44)
  Job Zones: already wide, shape = (923, 2)

Pivot complete.


### Merge all O\*NET wide tables together on SOC code

All six pivoted O\*NET datasets share `O*NET-SOC Code`. We merge them into a single wide feature table before standardizing SOC codes.

In [75]:
# Merge all O*NET wide tables
onet_merged = skills_wide
for df in [abilities_wide, work_act_wide, work_ctx_wide, edu_wide, job_zones_wide]:
    onet_merged = onet_merged.merge(df, on='O*NET-SOC Code', how='outer')

log(f"O*NET merged shape: {onet_merged.shape}")
print(f"\nShape: {onet_merged.shape}")
onet_merged.head()

O*NET merged shape: (923, 639)

Shape: (923, 639)


,O*NET-SOC Code,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,Skill_Complex Problem Solving_LV,Skill_Coordination_IM,Skill_Coordination_LV,Skill_Critical Thinking_IM,...,EduTrExp_Required Level of Education_RL_12.0,EduTrExp_Required Level of Education_RL_2.0,EduTrExp_Required Level of Education_RL_3.0,EduTrExp_Required Level of Education_RL_4.0,EduTrExp_Required Level of Education_RL_5.0,EduTrExp_Required Level of Education_RL_6.0,EduTrExp_Required Level of Education_RL_7.0,EduTrExp_Required Level of Education_RL_8.0,EduTrExp_Required Level of Education_RL_9.0,Job Zone
0,11-1011.00,3.75,4.50,4.00,4.75,4.38,4.88,4.25,4.88,4.38,...,2.78,4.46,0.00,0.00,5.15,32.29,0.00,45.91,3.94,5
1,11-1011.03,3.75,3.88,4.00,4.00,4.00,4.12,3.75,3.88,4.12,...,0.00,0.00,0.00,0.00,0.00,18.52,0.00,74.07,7.41,5
2,11-1021.00,3.62,3.75,4.00,4.12,3.62,3.88,3.88,4.00,3.88,...,0.00,28.76,5.72,21.59,5.99,27.30,0.14,0.15,0.00,4
3,11-1031.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,11-2011.00,3.25,4.12,4.12,4.12,3.50,3.88,3.50,4.12,4.00,...,0.00,9.82,0.00,7.67,8.04,60.02,2.42,5.87,0.00,4


## Step 3: Standardize SOC Codes to 6-Digit Format

O\*NET uses 8-digit codes (e.g., `11-1011.00`) while AIOE and BLS use 6-digit codes (e.g., `11-1011`). We truncate all O\*NET codes to 6 digits.

Since multiple 8-digit O\*NET codes can map to the same 6-digit SOC code (e.g., `15-1251.00` and `15-1251.01` both become `15-1251`), we aggregate by taking the **mean** of feature values for occupations that collapse to the same 6-digit code.

In [76]:
log("--- Step 3: Standardizing SOC codes to 6-digit format ---")

# Standardize O*NET SOC codes
onet_merged['SOC Code'] = onet_merged['O*NET-SOC Code'].apply(standardize_soc)
onet_merged = onet_merged.drop(columns=['O*NET-SOC Code'])

# Aggregate duplicate 6-digit SOC codes (mean of numeric columns)
n_before = onet_merged['SOC Code'].nunique()
numeric_cols = onet_merged.select_dtypes(include=np.number).columns.tolist()
onet_wide = onet_merged.groupby('SOC Code')[numeric_cols].mean().reset_index()
n_after = onet_wide['SOC Code'].nunique()

log(f"  O*NET: {n_before} unique 6-digit SOC codes aggregated to {n_after} rows")

# Standardize AIOE SOC codes (already 6-digit, but ensure clean format)
aioe['SOC Code'] = aioe['SOC Code'].apply(standardize_soc)
lmoe['SOC Code'] = lmoe['SOC Code'].apply(standardize_soc)
igoe['SOC Code'] = igoe['SOC Code'].apply(standardize_soc)

print(f"\nO*NET wide shape after SOC standardization: {onet_wide.shape}")
onet_wide.head()

--- Step 3: Standardizing SOC codes to 6-digit format ---
  O*NET: 798 unique 6-digit SOC codes aggregated to 798 rows

O*NET wide shape after SOC standardization: (798, 639)


,SOC Code,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,Skill_Complex Problem Solving_LV,Skill_Coordination_IM,Skill_Coordination_LV,Skill_Critical Thinking_IM,...,EduTrExp_Required Level of Education_RL_12.0,EduTrExp_Required Level of Education_RL_2.0,EduTrExp_Required Level of Education_RL_3.0,EduTrExp_Required Level of Education_RL_4.0,EduTrExp_Required Level of Education_RL_5.0,EduTrExp_Required Level of Education_RL_6.0,EduTrExp_Required Level of Education_RL_7.0,EduTrExp_Required Level of Education_RL_8.0,EduTrExp_Required Level of Education_RL_9.0,Job Zone
0,11-1011,3.75,4.19,4.00,4.375,4.19,4.50,4.00,4.38,4.25,...,1.39,2.23,0.00,0.00,2.575,25.405,0.00,59.99,5.675,5.0
1,11-1021,3.62,3.75,4.00,4.120,3.62,3.88,3.88,4.00,3.88,...,0.00,28.76,5.72,21.59,5.990,27.300,0.14,0.15,0.000,4.0
2,11-1031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
3,11-2011,3.25,4.12,4.12,4.120,3.50,3.88,3.50,4.12,4.00,...,0.00,9.82,0.00,7.67,8.040,60.020,2.42,5.87,0.000,4.0
4,11-2021,3.88,4.12,3.88,4.120,3.62,3.88,3.50,3.75,3.88,...,0.00,3.53,0.00,2.80,3.030,55.760,0.00,24.36,0.000,4.0


## Step 4: Prepare BLS Datasets for Merging

### BLS Occupational Employment and Wage Statistics
This file contains data at multiple geographic levels. We filter to **national-level** data only (`AREA_TITLE == "U.S."` or `AREA_TYPE == 1`) and keep the **"detailed"** occupation level. Wage columns are stored as strings (due to suppressed values like `#` and `**`), so we coerce them to numeric.

### BLS Current Population Survey
This dataset uses **occupation names** rather than SOC codes. We'll need to match it to our SOC-coded data via the occupation title from the AIOE dataset.

In [77]:
log("--- Step 4: Preparing BLS datasets ---")

# --- BLS Wages: filter to national-level, detailed occupations ---
# Check what area/level columns exist
print("Unique AREA_TYPE values:", bls_wages['AREA_TYPE'].unique()[:10] if 'AREA_TYPE' in bls_wages.columns else "N/A")
print("Sample AREA_TITLE values:", bls_wages['AREA_TITLE'].unique()[:5] if 'AREA_TITLE' in bls_wages.columns else "N/A")
print("Unique O_GROUP values:", bls_wages['O_GROUP'].unique() if 'O_GROUP' in bls_wages.columns else "N/A")

--- Step 4: Preparing BLS datasets ---
Unique AREA_TYPE values: [1 2 3 4 6]
Sample AREA_TITLE values: ['U.S.' 'Alabama' 'Alaska' 'Arizona' 'Arkansas']
Unique O_GROUP values: ['total' 'major' 'minor' 'broad' 'detailed']


In [78]:
  # Filter BLS wages to national level, detailed occupations, cross-industry
  bls_national = bls_wages[
      (bls_wages['AREA_TITLE'] == 'U.S.') &
      (bls_wages['O_GROUP'] == 'detailed') &
      (bls_wages['NAICS_TITLE'] == 'Cross-industry')
  ].copy()

  log(f"  BLS wages filtered: {bls_wages.shape[0]} rows -> {bls_national.shape[0]} national detailed rows")

  # Standardize SOC code
  bls_national['SOC Code'] = bls_national['OCC_CODE'].apply(standardize_soc)

  # Convert wage/employment columns to numeric (coerce non-numeric like '#', '**' to NaN)
  wage_cols = ['TOT_EMP', 'A_MEAN', 'A_MEDIAN', 'H_MEAN', 'H_MEDIAN', 'A_PCT10', 'A_PCT25', 'A_PCT75', 'A_PCT90']
  for col in wage_cols:
      if col in bls_national.columns:
          bls_national[col] = pd.to_numeric(bls_national[col], errors='coerce')

  # Keep only the columns we need
  bls_keep_cols = ['SOC Code', 'OCC_TITLE', 'TOT_EMP', 'A_MEDIAN', 'A_MEAN']
  bls_clean = bls_national[[c for c in bls_keep_cols if c in bls_national.columns]].copy()
  bls_clean = bls_clean.rename(columns={
      'OCC_TITLE': 'Occupation Title (BLS)',
      'TOT_EMP': 'total_employment',
      'A_MEDIAN': 'median_annual_wage',
      'A_MEAN': 'mean_annual_wage'
  })

  # Drop duplicate SOC codes (keep first — national cross-industry should be unique, but just in case)
  bls_clean = bls_clean.drop_duplicates(subset='SOC Code', keep='first')

  log(f"  BLS wages clean shape: {bls_clean.shape}")
  print(f"\nShape: {bls_clean.shape}")
  bls_clean.head()

  BLS wages filtered: 414437 rows -> 831 national detailed rows
  BLS wages clean shape: (831, 5)

Shape: (831, 5)


,SOC Code,Occupation Title (BLS),total_employment,median_annual_wage,mean_annual_wage
4,11-1011,Chief Executives,211850,206420.0,262930.0
6,11-1021,General and Operations Managers,3584420,102950.0,133120.0
8,11-1031,Legislators,26510,44810.0,67390.0
11,11-2011,Advertising and Promotions Managers,21100,126960.0,149270.0
13,11-2021,Marketing Managers,384980,161030.0,171520.0


In [79]:
  # --- BLS CPS: demographics by occupation ---
  # CPS uses occupation names, not SOC codes. We'll match to AIOE occupation titles.
  # First, build a lookup from occupation title -> SOC code using the AIOE file
  title_to_soc = dict(zip(
      aioe['Occupation Title'].str.strip().str.lower(),
      aioe['SOC Code']
  ))

  # Try exact match first (lowercase, stripped)
  bls_cps['occupation_clean'] = bls_cps['occupation'].str.strip().str.lower()

  # Map CPS occupation names to SOC codes
  bls_cps['SOC Code'] = bls_cps['occupation_clean'].map(title_to_soc)

  matched = bls_cps['SOC Code'].notna().sum()
  unmatched = bls_cps['SOC Code'].isna().sum()
  log(f"  BLS CPS: {matched} occupations matched to SOC codes, {unmatched} unmatched")

  if unmatched > 0:
      print(f"\nUnmatched CPS occupations (first 20):")
      print(bls_cps.loc[bls_cps['SOC Code'].isna(), 'occupation'].head(20).tolist())

  # Keep only matched rows with demographic columns — include pct_white
  cps_cols = ['SOC Code', 'total_employed_thousands', 'pct_women', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic']
  cps_available = [c for c in cps_cols if c in bls_cps.columns]
  bls_cps_clean = bls_cps.loc[bls_cps['SOC Code'].notna(), cps_available].copy()

  log(f"  BLS CPS clean shape: {bls_cps_clean.shape}")
  print(f"\nShape: {bls_cps_clean.shape}")
  print(f"Columns: {list(bls_cps_clean.columns)}")
  bls_cps_clean.head()

  BLS CPS: 220 occupations matched to SOC codes, 154 unmatched

Unmatched CPS occupations (first 20):
['Total, 16 years and over', 'Legislators', 'Public relations and fundraising managers', 'Facilities managers', 'Education and childcare administrators', 'Architectural and engineering managers', 'Funeral home managers', 'Entertainment and recreation managers', 'Personal service managers, all other', 'Claims adjusters, appraisers, examiners, and investigators', 'Compliance officers', 'Human resources workers', 'Project management specialists', 'Meeting, convention, and event planners', 'Property appraisers and assessors', 'Financial and investment analysts', 'Credit counselors and loan officers', 'Tax examiners and collectors, and revenue agents', 'Other financial specialists', 'Software developers']
  BLS CPS clean shape: (220, 7)

Shape: (220, 7)
Columns: ['SOC Code', 'total_employed_thousands', 'pct_women', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic']


,SOC Code,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
1,11-1011,1802,33.0,85.3,5.6,7.0,8.4
2,11-1021,1379,34.4,83.0,8.4,4.0,14.1
4,11-2011,61,53.7,72.0,16.0,11.9,6.2
5,11-2021,669,59.4,80.0,5.7,9.9,11.5
6,11-2022,563,33.2,85.3,5.0,6.1,13.5


## Step 5: Merge All Datasets on SOC Code

We now merge everything together:
1. Start with the O\*NET wide feature table
2. Merge AIOE, LMOE, and IGOE target scores
3. Merge BLS wage/employment data
4. Merge BLS CPS demographic shares

We use **inner joins** with the AIOE scores (since those define our target occupations) and **left joins** for BLS data (which may not cover all occupations). We also flag SOC codes that don't match across datasets.

In [80]:
log("--- Step 5: Merging all datasets on SOC Code ---")

# Flag SOC codes that exist in some datasets but not others
onet_socs = set(onet_wide['SOC Code'])
aioe_socs = set(aioe['SOC Code'])
bls_wage_socs = set(bls_clean['SOC Code'])
bls_cps_socs = set(bls_cps_clean['SOC Code'])

print("SOC code coverage:")
print(f"  O*NET:     {len(onet_socs)} unique SOC codes")
print(f"  AIOE:      {len(aioe_socs)} unique SOC codes")
print(f"  BLS Wages: {len(bls_wage_socs)} unique SOC codes")
print(f"  BLS CPS:   {len(bls_cps_socs)} unique SOC codes")

# Flag mismatches
in_aioe_not_onet = aioe_socs - onet_socs
in_onet_not_aioe = onet_socs - aioe_socs
in_aioe_not_bls = aioe_socs - bls_wage_socs

if in_aioe_not_onet:
  msg = f"  WARNING: {len(in_aioe_not_onet)} SOC codes in AIOE but NOT in O*NET: {sorted(list(in_aioe_not_onet))[:10]}..."
  log(msg)
  print(msg)
if in_onet_not_aioe:
  msg = f"  NOTE: {len(in_onet_not_aioe)} SOC codes in O*NET but NOT in AIOE (will be dropped on merge)"
  log(msg)
  print(msg)
if in_aioe_not_bls:
  msg = f"  NOTE: {len(in_aioe_not_bls)} SOC codes in AIOE but NOT in BLS wages"
  log(msg)
  print(msg)

--- Step 5: Merging all datasets on SOC Code ---
SOC code coverage:
  O*NET:     798 unique SOC codes
  AIOE:      774 unique SOC codes
  BLS Wages: 831 unique SOC codes
  BLS CPS:   220 unique SOC codes
  NOTE: 115 SOC codes in O*NET but NOT in AIOE (will be dropped on merge)
  NOTE: 115 SOC codes in O*NET but NOT in AIOE (will be dropped on merge)
  NOTE: 101 SOC codes in AIOE but NOT in BLS wages
  NOTE: 101 SOC codes in AIOE but NOT in BLS wages


In [81]:
# Merge: start with AIOE (defines our target occupations)
# Keep Occupation Title from AIOE as the canonical title
df = aioe[['SOC Code', 'Occupation Title', 'AIOE']].copy()

# Merge LMOE
df = df.merge(lmoe[['SOC Code', 'Language Modeling AIOE']], on='SOC Code', how='left')

# Merge IGOE
df = df.merge(igoe[['SOC Code', 'Image Generation AIOE']], on='SOC Code', how='left')

log(f"  After merging AIOE+LMOE+IGOE: {df.shape}")

# Merge O*NET features
df = df.merge(onet_wide, on='SOC Code', how='inner')
log(f"  After merging O*NET features: {df.shape}")

# Merge BLS wages
df = df.merge(bls_clean.drop(columns=['Occupation Title (BLS)'], errors='ignore'), on='SOC Code', how='left')
log(f"  After merging BLS wages: {df.shape}")

# Merge BLS CPS demographics
df = df.merge(bls_cps_clean, on='SOC Code', how='left')
log(f"  After merging BLS CPS demographics: {df.shape}")

print(f"\nFull merged dataset: {df.shape}")
df.head()

  After merging AIOE+LMOE+IGOE: (774, 5)
  After merging O*NET features: (683, 643)
  After merging BLS wages: (683, 646)
  After merging BLS CPS demographics: (683, 652)

Full merged dataset: (683, 652)


,SOC Code,Occupation Title,AIOE,Language Modeling AIOE,Image Generation AIOE,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,...,Job Zone,total_employment,median_annual_wage,mean_annual_wage,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,11-1011,Chief Executives,1.334246,1.308912,1.309602,3.75,4.19,4.00,4.375,4.19,...,5.0,211850.0,206420.0,262930.0,1802.0,33.0,85.3,5.6,7.0,8.4
1,11-1021,General and Operations Managers,0.574877,0.677615,0.231884,3.62,3.75,4.00,4.120,3.62,...,4.0,3584420.0,102950.0,133120.0,1379.0,34.4,83.0,8.4,4.0,14.1
2,11-2011,Advertising and Promotions Managers,1.294387,1.217224,1.614598,3.25,4.12,4.12,4.120,3.50,...,4.0,21100.0,126960.0,149270.0,61.0,53.7,72.0,16.0,11.9,6.2
3,11-2021,Marketing Managers,1.315032,1.203774,1.610358,3.88,4.12,3.88,4.120,3.62,...,4.0,384980.0,161030.0,171520.0,669.0,59.4,80.0,5.7,9.9,11.5
4,11-2022,Sales Managers,1.266280,1.293821,1.009998,3.75,3.88,4.00,4.000,3.75,...,4.0,603710.0,138060.0,160930.0,563.0,33.2,85.3,5.0,6.1,13.5


## Step 6: Drop Columns with >20% Missing Values

Columns with too many missing values provide little signal and can distort imputation. We remove any column where more than 20% of rows have missing data.

In [82]:
log("--- Step 6: Dropping columns with >20% missing values ---")

  # Protect BLS demographic and wage/employment columns from being dropped —
  # they have legitimately lower coverage (CPS uses occupation names, not SOC codes)
  # but are still valuable predictors when available.
protected_cols = {
    'pct_women', 'pct_black', 'pct_hispanic', 'pct_asian', 'pct_white',
    'total_employment', 'median_annual_wage', 'mean_annual_wage',
    'total_employed_thousands',
    'AIOE', 'Language Modeling AIOE', 'Image Generation AIOE',
    'SOC Code', 'Occupation Title'
}

n_rows = len(df)
missing_pct = df.isnull().sum() / n_rows
high_missing = missing_pct[missing_pct > 0.20]

# Only drop columns that are NOT protected
droppable = [col for col in high_missing.index if col not in protected_cols]
kept_despite_missing = [col for col in high_missing.index if col in protected_cols]

cols_before = df.shape[1]
if droppable:
    print(f"Columns with >20% missing being dropped ({len(droppable)} total):")
    for col in droppable:
        print(f"  {col}: {missing_pct[col]:.1%} missing")
    df = df.drop(columns=droppable)
else:
    print("No droppable columns exceed the 20% missing threshold.")

if kept_despite_missing:
    print(f"\nProtected columns kept despite >20% missing ({len(kept_despite_missing)}):")
    for col in kept_despite_missing:
        print(f"  {col}: {missing_pct[col]:.1%} missing (protected — BLS/target column)")

cols_after = df.shape[1]
log(f"  Dropped {cols_before - cols_after} columns with >20% missing. Columns: {cols_before} -> {cols_after}")
if kept_despite_missing:
    log(f"  Kept {len(kept_despite_missing)} protected BLS/target columns despite >20% missing")

print(f"\nShape after dropping high-missing columns: {df.shape}")
df.head()

--- Step 6: Dropping columns with >20% missing values ---
Columns with >20% missing being dropped (2 total):
  EduTrExp_Job-Related Professional Certification_IM_nan: 37.6% missing
  EduTrExp_Job-related Apprenticeship_IM_nan: 37.8% missing

Protected columns kept despite >20% missing (6):
  total_employed_thousands: 70.1% missing (protected — BLS/target column)
  pct_women: 71.6% missing (protected — BLS/target column)
  pct_white: 71.6% missing (protected — BLS/target column)
  pct_black: 71.6% missing (protected — BLS/target column)
  pct_asian: 71.6% missing (protected — BLS/target column)
  pct_hispanic: 71.6% missing (protected — BLS/target column)
  Dropped 2 columns with >20% missing. Columns: 652 -> 650
  Kept 6 protected BLS/target columns despite >20% missing

Shape after dropping high-missing columns: (683, 650)


,SOC Code,Occupation Title,AIOE,Language Modeling AIOE,Image Generation AIOE,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,...,Job Zone,total_employment,median_annual_wage,mean_annual_wage,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,11-1011,Chief Executives,1.334246,1.308912,1.309602,3.75,4.19,4.00,4.375,4.19,...,5.0,211850.0,206420.0,262930.0,1802.0,33.0,85.3,5.6,7.0,8.4
1,11-1021,General and Operations Managers,0.574877,0.677615,0.231884,3.62,3.75,4.00,4.120,3.62,...,4.0,3584420.0,102950.0,133120.0,1379.0,34.4,83.0,8.4,4.0,14.1
2,11-2011,Advertising and Promotions Managers,1.294387,1.217224,1.614598,3.25,4.12,4.12,4.120,3.50,...,4.0,21100.0,126960.0,149270.0,61.0,53.7,72.0,16.0,11.9,6.2
3,11-2021,Marketing Managers,1.315032,1.203774,1.610358,3.88,4.12,3.88,4.120,3.62,...,4.0,384980.0,161030.0,171520.0,669.0,59.4,80.0,5.7,9.9,11.5
4,11-2022,Sales Managers,1.266280,1.293821,1.009998,3.75,3.88,4.00,4.000,3.75,...,4.0,603710.0,138060.0,160930.0,563.0,33.2,85.3,5.0,6.1,13.5


## Step 7: Drop Occupations with Total Employment Under 5,000

Very small occupations may not have reliable data and can be outliers. We filter to occupations with at least 5,000 total employed workers (from BLS data). Occupations missing employment data are kept (we don't want to lose them just because BLS didn't cover them).

In [83]:
log("--- Step 7: Dropping occupations with total employment < 5,000 ---")

rows_before = len(df)

if 'total_employment' in df.columns:
    # Keep rows where employment >= 5000 OR employment is missing (don't penalize missing BLS data)
    low_emp = df[(df['total_employment'].notna()) & (df['total_employment'] < 5000)]
    print(f"Occupations with employment < 5,000: {len(low_emp)}")
    if len(low_emp) > 0:
        print(f"Examples: {low_emp[['SOC Code', 'Occupation Title', 'total_employment']].head(10).to_string()}")

    df = df[(df['total_employment'] >= 5000) | (df['total_employment'].isna())]
else:
    print("total_employment column not found — skipping this filter.")

rows_after = len(df)
log(f"  Dropped {rows_before - rows_after} occupations with employment < 5,000. Rows: {rows_before} -> {rows_after}")

print(f"\nShape: {df.shape}")
df.head()

--- Step 7: Dropping occupations with total employment < 5,000 ---
Occupations with employment < 5,000: 52
Examples:     SOC Code                         Occupation Title  total_employment
20   11-9071                          Gaming Managers            4620.0
38   13-1074                   Farm Labor Contractors             410.0
61   15-2021                           Mathematicians            2220.0
69   17-2021                   Agricultural Engineers            1680.0
98   19-1011                        Animal Scientists            2470.0
109  19-2011                              Astronomers            1560.0
120  19-3032  Industrial-Organizational Psychologists            1050.0
122  19-3041                             Sociologists            2950.0
125  19-3092                              Geographers            1380.0
126  19-3093                               Historians            3140.0
  Dropped 52 occupations with employment < 5,000. Rows: 683 -> 631

Shape: (631, 650)


,SOC Code,Occupation Title,AIOE,Language Modeling AIOE,Image Generation AIOE,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,...,Job Zone,total_employment,median_annual_wage,mean_annual_wage,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,11-1011,Chief Executives,1.334246,1.308912,1.309602,3.75,4.19,4.00,4.375,4.19,...,5.0,211850.0,206420.0,262930.0,1802.0,33.0,85.3,5.6,7.0,8.4
1,11-1021,General and Operations Managers,0.574877,0.677615,0.231884,3.62,3.75,4.00,4.120,3.62,...,4.0,3584420.0,102950.0,133120.0,1379.0,34.4,83.0,8.4,4.0,14.1
2,11-2011,Advertising and Promotions Managers,1.294387,1.217224,1.614598,3.25,4.12,4.12,4.120,3.50,...,4.0,21100.0,126960.0,149270.0,61.0,53.7,72.0,16.0,11.9,6.2
3,11-2021,Marketing Managers,1.315032,1.203774,1.610358,3.88,4.12,3.88,4.120,3.62,...,4.0,384980.0,161030.0,171520.0,669.0,59.4,80.0,5.7,9.9,11.5
4,11-2022,Sales Managers,1.266280,1.293821,1.009998,3.75,3.88,4.00,4.000,3.75,...,4.0,603710.0,138060.0,160930.0,563.0,33.2,85.3,5.0,6.1,13.5


## Step 8: Impute Remaining Missing Numeric Values with Column Medians

After dropping high-missing columns and low-employment occupations, remaining missing values are imputed with the column median. This is a conservative approach that doesn't shift distributions much.

In [84]:
log("--- Step 8: Imputing missing numeric values with column medians ---")

numeric_cols = df.select_dtypes(include=np.number).columns
missing_before = df[numeric_cols].isnull().sum().sum()

for col in numeric_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

missing_after = df[numeric_cols].isnull().sum().sum()
log(f"  Imputed {missing_before - missing_after} missing values across {len(numeric_cols)} numeric columns")
log(f"  Remaining missing values: {missing_after}")

print(f"\nMissing values: {missing_before} -> {missing_after}")
print(f"Shape: {df.shape}")
df.head()

--- Step 8: Imputing missing numeric values with column medians ---
  Imputed 5236 missing values across 648 numeric columns
  Remaining missing values: 0

Missing values: 5236 -> 0
Shape: (631, 650)


,SOC Code,Occupation Title,AIOE,Language Modeling AIOE,Image Generation AIOE,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,...,Job Zone,total_employment,median_annual_wage,mean_annual_wage,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,11-1011,Chief Executives,1.334246,1.308912,1.309602,3.75,4.19,4.00,4.375,4.19,...,5.0,211850.0,206420.0,262930.0,1802.0,33.0,85.3,5.6,7.0,8.4
1,11-1021,General and Operations Managers,0.574877,0.677615,0.231884,3.62,3.75,4.00,4.120,3.62,...,4.0,3584420.0,102950.0,133120.0,1379.0,34.4,83.0,8.4,4.0,14.1
2,11-2011,Advertising and Promotions Managers,1.294387,1.217224,1.614598,3.25,4.12,4.12,4.120,3.50,...,4.0,21100.0,126960.0,149270.0,61.0,53.7,72.0,16.0,11.9,6.2
3,11-2021,Marketing Managers,1.315032,1.203774,1.610358,3.88,4.12,3.88,4.120,3.62,...,4.0,384980.0,161030.0,171520.0,669.0,59.4,80.0,5.7,9.9,11.5
4,11-2022,Sales Managers,1.266280,1.293821,1.009998,3.75,3.88,4.00,4.000,3.75,...,4.0,603710.0,138060.0,160930.0,563.0,33.2,85.3,5.0,6.1,13.5


## Step 9: Remove Near-Duplicate Columns (Pearson Correlation > 0.95)

Highly correlated features are redundant and can cause multicollinearity issues in models. We identify pairs of numeric columns with |correlation| > 0.95 and drop one from each pair.

In [85]:
log("--- Step 10: Standardizing predictor variables (X) ---")

# Columns that must NOT be scaled — targets, IDs, and all BLS columns
target_cols = ['AIOE', 'Language Modeling AIOE', 'Image Generation AIOE']
id_cols = ['SOC Code', 'Occupation Title']
bls_cols = [
    'total_employment', 'median_annual_wage', 'mean_annual_wage',
    'total_employed_thousands',
    'pct_women', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic'
]
do_not_scale = set(target_cols + id_cols + bls_cols)

# Snapshot every non-scalable numeric column BEFORE touching anything
raw_snapshot = {}
for col in target_cols + bls_cols:
    if col in df.columns:
        raw_snapshot[col] = df[col].copy()

# Only scale O*NET predictor columns
cols_to_scale = [c for c in df.select_dtypes(include=np.number).columns if c not in do_not_scale]

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# Restore targets from the original source dataframes (safest — bypasses any in-df drift)
aioe_map = aioe.drop_duplicates('SOC Code').set_index('SOC Code')['AIOE']
lmoe_map = lmoe.drop_duplicates('SOC Code').set_index('SOC Code')['Language Modeling AIOE']
igoe_map = igoe.drop_duplicates('SOC Code').set_index('SOC Code')['Image Generation AIOE']

df['AIOE'] = df['SOC Code'].map(aioe_map).values
df['Language Modeling AIOE'] = df['SOC Code'].map(lmoe_map).values
df['Image Generation AIOE'] = df['SOC Code'].map(igoe_map).values

# Restore BLS columns from the pre-scaling snapshot
for col in bls_cols:
    if col in raw_snapshot:
        df[col] = raw_snapshot[col].values

log(f"  Scaled {len(cols_to_scale)} O*NET predictor columns to mean=0, std=1")
log(f"  NOT scaled (restored from raw): {target_cols + [c for c in bls_cols if c in df.columns]}")

# --- Verification ---
print(f"Scaled {len(cols_to_scale)} O*NET predictor columns.\n")

print("Target columns (should be 0–1 range):")
for col in target_cols:
    if col in df.columns:
        print(f"  {col}: min={df[col].min():.4f}, max={df[col].max():.4f}, mean={df[col].mean():.4f}")

print("\nDemographic columns (should be 0–100 range):")
for col in ['pct_women', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic']:
    if col in df.columns:
        print(f"  {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")

print("\nWage/employment columns (raw values):")
for col in ['total_employment', 'median_annual_wage', 'mean_annual_wage', 'total_employed_thousands']:
    if col in df.columns:
        print(f"  {col}: min={df[col].min():.0f}, max={df[col].max():.0f}, mean={df[col].mean():.0f}")

print(f"\nShape: {df.shape}")
df.head()

--- Step 10: Standardizing predictor variables (X) ---
  Scaled 636 O*NET predictor columns to mean=0, std=1
  NOT scaled (restored from raw): ['AIOE', 'Language Modeling AIOE', 'Image Generation AIOE', 'total_employment', 'median_annual_wage', 'mean_annual_wage', 'total_employed_thousands', 'pct_women', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic']
Scaled 636 O*NET predictor columns.

Target columns (should be 0–1 range):
  AIOE: min=-2.6698, max=1.5261, mean=-0.0043
  Language Modeling AIOE: min=-1.8539, max=1.9256, mean=0.0038
  Image Generation AIOE: min=-3.6548, max=2.4210, mean=-0.0258

Demographic columns (should be 0–100 range):
  pct_women: min=0.90, max=97.30, mean=52.14
  pct_white: min=25.10, max=95.00, mean=76.71
  pct_black: min=0.00, max=43.50, mean=11.51
  pct_asian: min=0.00, max=67.50, mean=6.27
  pct_hispanic: min=3.50, max=66.00, mean=16.86

Wage/employment columns (raw values):
  total_employment: min=5000, max=3800250, mean=194222
  median_annual_wage: mi

,SOC Code,Occupation Title,AIOE,Language Modeling AIOE,Image Generation AIOE,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,...,Job Zone,total_employment,median_annual_wage,mean_annual_wage,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,11-1011,Chief Executives,1.334246,1.308912,1.309602,1.286664,1.399958,0.945346,1.359492,2.121455,...,1.757795,211850.0,206420.0,262930.0,1802.0,33.0,85.3,5.6,7.0,8.4
1,11-1021,General and Operations Managers,0.574877,0.677615,0.231884,1.041611,0.832858,0.945346,0.967783,0.969900,...,0.844858,3584420.0,102950.0,133120.0,1379.0,34.4,83.0,8.4,4.0,14.1
2,11-2011,Advertising and Promotions Managers,1.294387,1.217224,1.614598,0.344150,1.309737,1.200101,0.967783,0.727467,...,0.844858,21100.0,126960.0,149270.0,61.0,53.7,72.0,16.0,11.9,6.2
3,11-2021,Marketing Managers,1.315032,1.203774,1.610358,1.531718,1.309737,0.690592,0.967783,0.969900,...,0.844858,384980.0,161030.0,171520.0,669.0,59.4,80.0,5.7,9.9,11.5
4,11-2022,Sales Managers,1.266280,1.293821,1.009998,1.286664,1.000410,0.945346,0.783449,1.232535,...,0.844858,603710.0,138060.0,160930.0,563.0,33.2,85.3,5.0,6.1,13.5


## Step 10: Cleaning Summary

A full summary of the cleaning pipeline: how many occupations and features at each step, what was dropped, and why.

In [87]:
print("=" * 70)
print("CLEANING SUMMARY")
print("=" * 70)

for entry in cleaning_log:
    print(entry)

print("\n" + "=" * 70)
print("FINAL DATASET OVERVIEW")
print("=" * 70)
print(f"Shape: {df.shape[0]} occupations x {df.shape[1]} columns")
print(f"\nColumn categories:")
print(f"  ID columns: SOC Code, Occupation Title")
print(f"  Target variables (y): {[c for c in target_cols if c in df.columns]}")

remaining_predictors = [c for c in df.columns if c not in do_not_scale]
remaining_numeric_predictors = [c for c in cols_to_scale if c in df.columns]
print(f"  Predictor features (X): {len(remaining_numeric_predictors)} numeric columns (standardized)")

print(f"\nMissing values in final dataset: {df.isnull().sum().sum()}")
print(f"\nSample of final data:")
display(df.head(10))

CLEANING SUMMARY
--- Step 2: Pivoting O*NET files from long to wide ---
  Skills: pivoted from (62580, 16) to (894, 71)
  Abilities: pivoted from (92976, 16) to (894, 105)
  Work Activities: pivoted from (73308, 16) to (894, 83)
  Work Context: pivoted from (297676, 17) to (894, 339)
  Education/Training/Experience: pivoted from (37125, 16) to (878, 44)
  Job Zones: already wide, shape = (923, 2)
O*NET merged shape: (923, 639)
--- Step 3: Standardizing SOC codes to 6-digit format ---
  O*NET: 798 unique 6-digit SOC codes aggregated to 798 rows
--- Step 4: Preparing BLS datasets ---
  BLS wages filtered: 414437 rows -> 831 national detailed rows
  BLS wages clean shape: (831, 5)
  BLS CPS: 220 occupations matched to SOC codes, 154 unmatched
  BLS CPS clean shape: (220, 7)
--- Step 5: Merging all datasets on SOC Code ---
  NOTE: 115 SOC codes in O*NET but NOT in AIOE (will be dropped on merge)
  NOTE: 101 SOC codes in AIOE but NOT in BLS wages
  After merging AIOE+LMOE+IGOE: (774, 5)
  A

,SOC Code,Occupation Title,AIOE,Language Modeling AIOE,Image Generation AIOE,Skill_Active Learning_IM,Skill_Active Learning_LV,Skill_Active Listening_IM,Skill_Active Listening_LV,Skill_Complex Problem Solving_IM,...,Job Zone,total_employment,median_annual_wage,mean_annual_wage,total_employed_thousands,pct_women,pct_white,pct_black,pct_asian,pct_hispanic
0,11-1011,Chief Executives,1.334246,1.308912,1.309602,1.286664,1.399958,0.945346,1.359492,2.121455,...,1.757795,211850.0,206420.0,262930.0,1802.0,33.0,85.3,5.6,7.0,8.4
1,11-1021,General and Operations Managers,0.574877,0.677615,0.231884,1.041611,0.832858,0.945346,0.967783,0.969900,...,0.844858,3584420.0,102950.0,133120.0,1379.0,34.4,83.0,8.4,4.0,14.1
2,11-2011,Advertising and Promotions Managers,1.294387,1.217224,1.614598,0.344150,1.309737,1.200101,0.967783,0.727467,...,0.844858,21100.0,126960.0,149270.0,61.0,53.7,72.0,16.0,11.9,6.2
3,11-2021,Marketing Managers,1.315032,1.203774,1.610358,1.531718,1.309737,0.690592,0.967783,0.969900,...,0.844858,384980.0,161030.0,171520.0,669.0,59.4,80.0,5.7,9.9,11.5
4,11-2022,Sales Managers,1.266280,1.293821,1.009998,1.286664,1.000410,0.945346,0.783449,1.232535,...,0.844858,603710.0,138060.0,160930.0,563.0,33.2,85.3,5.0,6.1,13.5
5,11-3021,Computer and Information Systems Managers,1.059853,0.945070,1.275466,0.589204,1.000410,0.945346,0.783449,1.232535,...,0.844858,645970.0,171200.0,187990.0,646.0,25.1,72.9,4.5,20.0,7.0
6,11-3031,Financial Managers,1.446232,1.294551,1.330661,1.047894,1.210924,0.860428,1.167478,1.394157,...,1.149170,818620.0,161700.0,180470.0,1431.0,53.7,78.0,9.6,10.0,13.3
7,11-3051,Industrial Production Managers,0.508688,0.354297,0.654702,0.730581,0.723305,0.637518,0.655440,0.976634,...,0.540545,234380.0,121440.0,129180.0,345.0,22.8,81.7,4.8,10.5,13.9
8,11-3061,Purchasing Managers,1.423426,1.393102,1.298248,0.344150,0.832858,0.945346,0.783449,0.727467,...,0.844858,81240.0,139510.0,150630.0,222.0,46.8,85.9,5.1,6.0,16.1
9,11-3071,"Transportation, Storage, and Distribution Mana...",0.890792,0.904548,0.773122,1.164138,0.749082,0.817969,0.691282,1.232535,...,0.844858,213000.0,102010.0,116010.0,415.0,16.3,76.3,13.4,6.5,15.3


## Step 11: Save Final Dataset and Cleaning Log

In [88]:
# Save cleaned dataset
import os
os.makedirs(DATA_DIR, exist_ok=True)
output_path = f"{DATA_DIR}/data_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"  Shape: {df.shape}")

# Save cleaning log
log_path = f"{DATA_DIR}/cleaning_log.txt"
with open(log_path, 'w') as f:
    f.write("DATA CLEANING LOG\n")
    f.write("=" * 70 + "\n\n")
    for entry in cleaning_log:
        f.write(entry + "\n")
    f.write(f"\nFINAL DATASET: {df.shape[0]} occupations x {df.shape[1]} columns\n")
    f.write(f"Saved to: {output_path}\n")
print(f"Cleaning log saved to: {log_path}")

Cleaned dataset saved to: data/data_cleaned.csv
  Shape: (631, 650)
Cleaning log saved to: data/cleaning_log.txt
